# CAD-ViT — riset P1–P4

**Status P1–P4: NOT RUN.** Phase 3B memverifikasi cache input deterministik pada CPU; pilot cached belum dimulai. Notebook riset ini mendokumentasikan kandidat Phase 1/2 di branch `preof-kc` (snapshot HEAD `3a563c6`, source Phase 1/2 masih untracked); belum ada training, kurva, confusion matrix, ataupun hasil final test P1–P4. Sel kode hanya untuk introspeksi CPU, hash baca-saja opsional, dan kapasitas sistem. Perintah GPU/training adalah petunjuk manual untuk tahap berikutnya. Baseline [CAD_ViT.ipynb](CAD_ViT.ipynb), [reproduction/cadvit](reproduction/cadvit/), checkpoint, dan hasil historis tetap terpisah.

## Navigasi

1. [Paper dan masalah riset](#1-paper-dan-masalah-riset)
2. [Arsitektur serta formulasi](#2-arsitektur-serta-formulasi)
3. [Dataset dan provenance](#3-dataset-dan-provenance)
4. [Ablasi dan evaluasi](#4-ablasi-dan-evaluasi)
5. [Kesiapan GPU serta perintah KC](#5-kesiapan-gpu-serta-perintah-kc)
6. [Hasil dan batas ilmiah](#6-hasil-dan-batas-ilmiah)

## 1. Paper dan masalah riset

Y. Wang, X. Gou, W. Li, “CAD-ViT: Coordinate Attention-Enhanced Vision Transformer With Dilated Feature Fusion for Diabetic Retinopathy Staging,” *IEEE Access* 13 (2025), 91505–91514, DOI [10.1109/ACCESS.2025.3572455](https://doi.org/10.1109/ACCESS.2025.3572455). §III/Gambar 2–3 mendeskripsikan dilated feature fusion, coordinate attention, dan Transformer; §IV membahas APTOS2019 serta DDR.

**Spesifikasi paper:** alur sembilan blok pada Gambar 3, dilatasi 3/5/7/9, coordinate attention r=16, dan lima grade. **Asumsi reproduksi:** lebar kanal, stride/expansion, patch/token layout, Transformer depth/heads/FFN, dan classifier internal; lihat [audit fidelity](docs/ARCHITECTURE_FIDELITY_AUDIT.md). Tabel 1 paper: sekitar **7,2629 juta** parameter, sedangkan `CADViTCandidate`: **5.282.101**. V2 BG10 adalah ekstensi preprocessing, bukan langkah yang dapat diatribusikan ke paper. Penelitian ini menguji kandidat reproduksi parsial, bukan source resmi penulis.

**Pertanyaan eksploratori:** dapatkah perubahan terkontrol meningkatkan validation macro-F1 tanpa memilih metode pada test set?

| Komponen | Hipotesis yang akan diuji | Kontrol |
| --- | --- | --- |
| P1 local contrast | Jalur high-pass mungkin membawa detail yang berguna; belum ada bukti detail itu lesi. | Baseline vs P1 |
| P2 adaptive MFB | Bobot cabang per sampel mungkin lebih berguna daripada cabang tetap. | Baseline vs P2 |
| P3 boundary-conditioned modulation | Prediksi batas grade lokal mungkin menambah manfaat setelah supervisi ordinal dan gating biasa dikendalikan. | Probe vs full, dengan/tanpa P2 |
| P4 ordinal head | Auxiliary objective global mungkin membantu klasifikasi utama. | Baseline vs P4 |

Hipotesis nol setiap kontras: tidak ada peningkatan pada metrik validasi yang dipraspesifikasi. Semua hasil validation kelak bersifat eksploratori karena split itu juga memilih checkpoint awal.

## 2. Arsitektur serta formulasi

Diagram statis ini mengikuti `CADViTCandidate.block_names` dan letak modul di [novelty/cadvit_model.py](novelty/cadvit_model.py). Sel selanjutnya dapat membuat ulang diagram dari objek model aktual. Panah P1 berarti **penjumlahan setelah stem**, bukan penggantian input stem.

```mermaid
flowchart LR
  I[RGB 512×512] --> S[Stem] --> A[IRB-1] --> B[IRB+A-1 / P3]
  I -. P1 high-pass residual .-> S
  B --> C[IRB-2] --> D[IRB+A-2 / P3] --> E[EMB-1 / MFB P2]
  E --> F[IRB+A-3 / P3] --> G[EMB-2 / MFB P2]
  G --> H[IRB+A-4 / P3] --> J[EMB-3 / MFB P2]
  J --> K[Head 320] --> L[Average pool] --> M[5 logits]
  L -. P4 auxiliary .-> N[4 ordinal logits]
```

Empat P3 probe muncul pada IRB+A; tiga P2 gate pada MFB di EMB. P4 tidak mengganti classifier lima kelas.

### 2.1 Source impor dan grafik programatik

Sel ini menemukan root dan mengimpor implementasi yang sudah ada. Sel berikutnya menginstansiasi model **di CPU, tanpa checkpoint atau forward**, mengambil nama blok, dan memeriksa jumlah MFB/CA; tidak membaca citra atau final test.

In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "novelty/cadvit_model.py").is_file()
             and (p / "reproduction/cadvit/model.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run inside the novelty repository")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from IPython.display import Markdown, display
from reproduction.cadvit.model import (
    CADViTCandidate, CoordinateAttention, EnhancedMobileViTBlock, InvertedResidualBlock)
from novelty.cadvit_model import NovelCADViT, PRESETS, parameter_counts
from novelty.cadvit_pilot import CHECKPOINTS, PRIMARY_ABLATIONS
print("Root:", ROOT, "| pilot arms:", ", ".join(PRIMARY_ABLATIONS))

**Tujuan sel diagram.** Struktur baseline dibaca dari `block_names` dan tipe blok aktual; tiga lokasi MFB dan empat lokasi Coordinate Attention diverifikasi sebelum Mermaid ditampilkan. Ini pemeriksaan arsitektur statis, bukan forward pass atau bukti performa.

In [ ]:
base = CADViTCandidate()
blocks = list(zip(base.block_names, base.blocks))
mfb = [n for n, b in blocks if isinstance(b, EnhancedMobileViTBlock)]
ca = [n for n, b in blocks if isinstance(b, InvertedResidualBlock)
      and isinstance(b.attention, CoordinateAttention)]
assert len(blocks) == 9 and len(mfb) == 3 and len(ca) == 4
labels = ["Stem"] + [n + (" / P2" if n in mfb else " / P3" if n in ca else "")
                     for n, _ in blocks] + ["Head", "Pool", "5 logits"]
arrows = "\n".join(f'  n{i}["{v}"] --> n{i+1}["{labels[i+1]}"]'
                   for i, v in enumerate(labels[:-1]))
display(Markdown("Grafik dari nama blok aktual:\n\n```mermaid\nflowchart LR\n"
                 + arrows + "\n```"))
print("MFB:", mfb, "| CA:", ca)
del base

### 2.2 Formulasi dan hubungan P1–P4

**P1:** `D=I−AvgPool₃×₃(I)`; `S′(I)=S(I)+Conv₃×₃,stride₂(D)`. Proyeksi diawali nol sehingga output awal stem sama. Jalur ini mengekspos kontras lokal; keberhasilan mempertahankan lesi belum diketahui.

**P2:** untuk empat cabang asli `Fᵢ(X)` (dilatasi 3/5/7/9), `αᵢ=4·softmax(Linear(GAP(X)))ᵢ`; `MFB′=Fuse(Concat[αᵢFᵢ(X)])`. Gate nol pada awal memberi semua `αᵢ=1`, menjaga urutan serta bobot cabang/fusion. Ini kontrol ordinary adaptive gating.

**P3:** pada fitur `U` **sebelum** coordinate attention, `s=Linear(GAP(U))`, threshold berurutan `τ₀=a`, `τₖ=a+Σⱼ₌₁ᵏsoftplus(dⱼ)`, logit `zₖ=s−τₖ`, target train-only `tₖ=1[y>k]`. Empat probe menerima BCE. Probe-only mengeluarkan CA asli; full mengeluarkan `CA(U)·(1+tanh(Linear(sigmoid(z))))` per kanal. Modulasi diawali nol. Ini modulasi **keluaran fitur CA**, bukan perubahan gate spasial CA.

**P4:** head `Linear(h)` dari pooled feature `h∈R³²⁰` menghasilkan empat logit ordinal terhadap `tₖ` yang sama. Head lima kelas tetap primer.

**Loss eksploratori:** `L=CE(logit₅,y)+0,20·mean₄site BCE(z,t)[P3]+0,20·BCE(o,t)[P4]`. P3/P4 memakai target identik di kedalaman berbeda, sehingga kombinasi dapat menggandakan tekanan supervisi. P1/P2/P3/P4 bersama tidak mengisolasi kontribusi P3. Inisialisasi output identik dengan baseline tidak menjamin lintasan BatchNorm/optimisasi tetap identik.

### 2.3 Jumlah parameter

| Arm | Total parameter |
| --- | ---: |
| Baseline | 5.282.101 |
| P1 | 5.282.965 |
| P2 | 5.283.649 |
| P3 probe-only | 5.282.761 |
| P3 full | 5.285.961 |
| P4 | 5.283.385 |
| P2+P3 probe | 5.284.309 |
| P2+P3 full | 5.287.509 |
| P1+P2+P3+P4 | 5.289.657 |
| Paper Tabel 1, r=16 | ~7.262.900, angka dibulatkan |

Sel ini menghitung ulang tiap arm dari kelas yang diimpor. Tidak ada forward, GPU, bobot checkpoint, atau data.

In [ ]:
expected = dict(baseline=5282101, p1_only=5282965, p2_only=5283649,
                p3_probe_only=5282761, p3_only=5285961, p4_only=5283385,
                p2_p3_probe=5284309, p2_p3_full=5287509, p1234=5289657)
for arm in PRIMARY_ABLATIONS:
    model = CADViTCandidate() if arm == "baseline" else NovelCADViT(PRESETS[arm])
    counts = parameter_counts(model)
    assert counts["total"] == expected[arm], (arm, counts)
    print(f"{arm:16s}: total {counts['total']:,}; trainable {counts['trainable']:,}")
    del model

## 3. Dataset dan provenance

APTOS2019: **2.930 train / 366 validation**. DDR: **10.018 train / 1.252 validation**. Manifest seed 42 dibekukan di `~/Kel_13/cad_vit_runs/manifests/<dataset>_seed42.csv`. `load_train_val_manifest` memverifikasi manifest+sidecar, hash semua citra train/validation, serta pemisahan group/konten; baris **test dilewati sebelum path/label digunakan**. Tidak ada split baru, oversampling, atau weighted CE pada pilot.

Runner memanggil `preprocess_fundus_v2` dengan `PREPROCESS_CONFIG` dan `BG_V2_CONFIG`; train lalu mendapat horizontal/vertical flip p=0,5 dan affine degrees=15/shear=10 bilinear fill=0, diikuti RGB CHW float32/255. Validation deterministik tanpa augmentasi. Transformasi geometri sesuai runner CAD-ViT yang diekstrak; kesetaraan tepat dengan implementasi historis V2 belum terbukti.

| Dataset | SHA-256 manifest CSV | Checkpoint arsip V2 BG10 dan SHA-256 | Epoch zero-based |
| --- | --- | --- | ---: |
| APTOS2019 | `73d1546e3c2fd2bf9e3d0fd10af914cb523d28e4bc8e1f25212af454d4be8854` | `experiment_archive_80ep_20261105/APTOS/best.pt` · `e72573f0f1c26baac53e18d950076845952da503261e212021ff6320c9ae72d5` | 68 |
| DDR | `7bc3e3916e012bc46fcd9cd69aa469e0daf3229e1273f1d8ecb6054899bf6eb1` | `experiment_archive_80ep_20261105/DDR/best.pt` · `844665f0c597142915286b727a21f1d2223f43e0f057d083b6c1dd035c180ac1` | 71 |

Checkpoint hash, model mapping, V2 processing, manifest hash, dataset, seed, LR, batch, dan metadata recovery DDR menjadi guard runner. DDR checkpoint memiliki riwayat mixed weighted-FP32 recovery; pilot baru bukan kelanjutan optimizer tersebut. Hash membuktikan identitas byte dengan nilai rujukan, tidak sendiri membuktikan seluruh lineage.

### 3.1 Hash check baca-saja, opt-in

Default sel ini **SKIPPED**. Bila diperlukan, aktifkan flag dan `CADVIT_ARTIFACT_ROOT` untuk menghash dua manifest CSV train/validation dan dua checkpoint di atas; tidak membaca citra/prediksi test. Preflight CLI menjalankan validasi train/validation yang lebih lengkap.

In [ ]:
VERIFY_LOCAL_PROVENANCE = False
if VERIFY_LOCAL_PROVENANCE:
    import os
    from novelty.cadvit_training import file_sha256
    from novelty.preof_phase1 import SPECS
    ARTIFACT_ROOT = Path(os.environ["CADVIT_ARTIFACT_ROOT"]).expanduser().resolve()
    for dataset in ("APTOS2019", "DDR"):
        sources = (
            (ARTIFACT_ROOT / "cad_vit_runs" / "manifests" / f"{dataset}_seed42.csv",
             SPECS[dataset]["manifest_sha256"]),
            (ARTIFACT_ROOT / CHECKPOINTS[dataset]["relative"], CHECKPOINTS[dataset]["sha256"]),
        )
        for path, expected_sha in sources:
            observed = file_sha256(path)
            assert observed == expected_sha, (dataset, path, observed)
            print(dataset, path.name, "SHA-256 PASS", observed)
else:
    print("SKIPPED: set VERIFY_LOCAL_PROVENANCE=True and CADVIT_ARTIFACT_ROOT.")

## 4. Ablasi dan evaluasi

| Arm CLI | P1 | P2 | P3 probe BCE | P3 modulation | P4 BCE | Tujuan |
| --- | :---: | :---: | :---: | :---: | :---: | --- |
| `baseline` | — | — | — | — | — | Kontrol asli |
| `p1_only` | ✓ | — | — | — | — | Detail |
| `p2_only` | — | ✓ | — | — | — | Gating biasa |
| `p3_probe_only` | — | — | ✓ | — | — | Supervisi lokal |
| `p3_only` | — | — | ✓ | ✓ | — | Modulasi di atas probe |
| `p4_only` | — | — | — | — | ✓ | Supervisi ordinal global |
| `p2_p3_probe` | — | ✓ | ✓ | — | — | Gating + probe |
| `p2_p3_full` | — | ✓ | ✓ | ✓ | — | Modulasi di atas gating + probe |
| `p1234` | ✓ | ✓ | ✓ | ✓ | ✓ | Kombinasi eksploratori |

Kontras utama: `p2_p3_full` vs `p2_p3_probe`; kontras kedua `p3_only` vs `p3_probe_only`. Batas lain harus sama dalam tiap dataset: checkpoint, manifest, seed, microbatch, augmentasi, optimizer, epoch, aturan seleksi. Jangan memilih arm dengan final test.

Pilot: seed 42, AdamW LR 0,000186, weight decay 0, FP32, grad clip 1, effective batch 16, microbatch pembagi 16, **1–5 epoch**. Per epoch tersimpan loss, accuracy, macro/weighted F1, metrik per kelas (precision/recall/F1/specificity/support), confusion matrix true-row/predicted-column, QWK, grade MAE, waktu, dan peak allocated VRAM. `best.pt`: validation macro-F1 tertinggi; tie persis: primary validation CE loss lebih rendah, lalu epoch lebih awal. `last.pt`, config, JSONL history, summary disimpan luar repo. Resume menolak config/history/best/last atau RNG state yang tidak cocok.

## 5. Kesiapan GPU serta perintah KC

Snapshot infrastruktur: kedua RTX 3060 bersama sedang dipakai; `/mnt/home` ~99% terpakai, ~22 GiB bebas. Ini bukan angka tetap. **Synthetic 512×512 GPU forward/backward batch 1 masih PENDING**; CPU 512×512 dan GPU 64×64 tidak membuktikan kebutuhan memori GPU penuh. Jangan mengganggu proses pengguna lain. Tidak ada ambang VRAM aman yang telah diukur; cek ketersediaan aktual serta disk sebelum smoke/pilot.

`CUDA_VISIBLE_DEVICES=<approved_index>` memilih GPU fisik; di dalam proses ia menjadi `cuda:0`. `--device cuda` tidak fallback ke CPU. `preflight` memeriksa hash train/validation dan checkpoint tanpa inference; `smoke` hanya forward/backward **tensor sintetis 512×512**, tanpa optimizer step/data; `pilot`/`resume` adalah training manual yang belum diizinkan pada tahap ini.

**Phase 3B:** run APTOS lama tanpa cache di `~/Kel_13/cadvit_novelty_pilots/APTOS2019/p2_p3_probe_seed42_e2_mb1_eff16_fp32` tetap utuh dan tidak di-resume. Run cached baru memakai `~/Kel_13/cadvit_novelty_pilots_cached` dan cache eksternal `~/Kel_13/cadvit_v2_cache`. Dua blok perintah di bawah sekarang mengaktifkan cache.

### 5.1 Snapshot kapasitas baca-saja

Sel ini hanya membaca disk dan `nvidia-smi`; tidak memulai workload GPU. Tidak memakai batas siap-training yang belum terukur.

In [ ]:
import os, shutil, subprocess
ARTIFACT_ROOT = Path(os.environ.get("CADVIT_ARTIFACT_ROOT",
                                    "/mnt/home/probation-03/Kel_13")).expanduser().resolve()
usage = shutil.disk_usage(ARTIFACT_ROOT)
print(f"Filesystem free: {usage.free / 2**30:.2f} GiB ({usage.free / usage.total:.1%})")
probe = subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.total,memory.used,memory.free,utilization.gpu",
     "--format=csv,noheader,nounits"], capture_output=True, text=True, check=False)
print(probe.stdout.strip() if probe.returncode == 0 else "nvidia-smi unavailable")
print("READ-ONLY snapshot; 512×512 GPU smoke is still required.")

### 5.2 Cache deterministik V2 BG10 dan audit Phase 3C

Cache menyimpan hanya output `preprocess_fundus_v2`: RGB `uint8` [512,512,3] tanpa kehilangan piksel. Flip/affine train-only tetap sesudah pembacaan cache dan acak tiap epoch; validation tetap deterministik. `--cache-mode off` adalah jalur langsung. Key/metadata memuat SHA citra terverifikasi, SHA manifest, konfigurasi V2, hash source preprocessing/cache, versi Pillow/OpenCV/NumPy, hash NPY dan piksel, shape, dtype. Entry yang hilang/rusak dibangun ulang di bawah lock Linux `fcntl` yang dilepas otomatis bila writer mati. Resume menolak identitas cache/protokol/source berbeda; preflight CPU tidak menginisialisasi CUDA.

Audit mandiri mengambil gambar **train dan validation pertama untuk setiap grade 0–4** pada APTOS2019 serta DDR. Array RGB dan tensor float32 cache/direct identik; RNG augmentasi terkontrol identik, perubahan lintas epoch tetap terjadi, validation berulang identik. Entry hilang, metadata stale/rusak, publikasi terputus, serta dua pembaca konkuren lulus pemeriksaan di cache sementara. Tidak ada cache penuh, inferensi GPU, atau test set.

| Dataset | Gambar train+val | Direct | Cold fill | Warm read | Hasil |
| --- | ---: | ---: | ---: | ---: | --- |
| APTOS2019 | 10 | 2,065 s | 2,525 s | 0,01234 s | exact |
| DDR | 10 | 0,722 s | 1,239 s | 0,01242 s | exact |

Ini timing CPU serial pada subset kecil; cold fill diukur setelah direct decoding sehingga page cache OS mungkin sudah hangat. Estimasi disk terkoreksi: NPY 786.560 byte memakai 790.528 byte teralokasi, JSON memakai 4.096 byte lagi. Budget konservatif **798.720 byte per SHA unik** memberi APTOS **2,368 GiB** untuk 3.183 SHA unik dan DDR **8,315 GiB** untuk 11.178 SHA unik; batas jika tiap baris unik adalah **2,452/8,383 GiB**. Reserve **8 GiB free** adalah kebijakan, bukan kebutuhan VRAM/training terukur. Pada audit, ruang bersama turun menjadi ~13,94 GiB; DDR full warm-up **BLOCKED**, dan APTOS memerlukan tinjauan ulang kapasitas serta persetujuan. Tiga namespace benchmark APTOS kecil dari Phase 3B (18 entry, 14.158.080 byte NPY) tetap dipertahankan di luar repo; source yang diperbaiki memakai identitas cache baru. [Protokol dan perintah Phase 3C](docs/CADVIT_NOVELTY_PROTOCOL.md) mencatat rinciannya.

Perintah full warm-up di bawah hanya untuk tahap setelah persetujuan terpisah. Smoke GPU 512×512 dan pilot masih menunggu kapasitas perangkat bersama.

### 5.3 Perintah manual APTOS2019

Jalankan dari root worktree KC; pilih `GPU` hanya setelah memeriksa pengguna lain. **Jangan jalankan full warm-up, pilot, atau resume sekarang.** Untuk arm lain ubah `--ablation`; runner memberi direktori independen.

```bash
cd /mnt/home/probation-03/Kel_13/Diabetic-Rethinography-Novelty
ROOT=/mnt/home/probation-03/Kel_13
OUT="$ROOT/cadvit_novelty_pilots_cached"
CACHE="$ROOT/cadvit_v2_cache"
read -r -p "Approved free GPU index (0 or 1): " GPU
../.venv/bin/python -m novelty.cadvit_pilot preflight --artifact-root "$ROOT" --output-root "$OUT" --cache-mode on --cache-root "$CACHE" --dataset APTOS2019 --ablation p2_p3_probe --micro-batch 1 --epochs 2
# Full cache warm-up only after separate approval (not run in Phase 3B):
../.venv/bin/python -m novelty.cadvit_pilot cache-warmup --artifact-root "$ROOT" --output-root "$OUT" --cache-mode on --cache-root "$CACHE" --warm-split both --warm-limit 3296 --dataset APTOS2019 --ablation p2_p3_probe --micro-batch 1 --epochs 2
CUDA_VISIBLE_DEVICES="$GPU" ../.venv/bin/python -m novelty.cadvit_pilot smoke --artifact-root "$ROOT" --output-root "$OUT" --cache-mode on --cache-root "$CACHE" --dataset APTOS2019 --ablation p2_p3_probe --micro-batch 1 --epochs 2 --device cuda
# Hanya setelah gate 512×512 GPU dan review ilmiah:
CUDA_VISIBLE_DEVICES="$GPU" ../.venv/bin/python -m novelty.cadvit_pilot pilot --artifact-root "$ROOT" --output-root "$OUT" --cache-mode on --cache-root "$CACHE" --dataset APTOS2019 --ablation p2_p3_probe --micro-batch 1 --epochs 2 --device cuda
CUDA_VISIBLE_DEVICES="$GPU" ../.venv/bin/python -m novelty.cadvit_pilot resume --artifact-root "$ROOT" --output-root "$OUT" --cache-mode on --cache-root "$CACHE" --dataset APTOS2019 --ablation p2_p3_probe --micro-batch 1 --epochs 2 --device cuda
```

### 5.4 Perintah manual DDR

DDR memakai manifest/checkpoint berbeda dan output subdirektori sendiri. Recovery checkpoint DDR weighted FP32; pilot baru CE biasa. **Jangan jalankan full warm-up, pilot, atau resume sekarang.**

```bash
cd /mnt/home/probation-03/Kel_13/Diabetic-Rethinography-Novelty
ROOT=/mnt/home/probation-03/Kel_13
OUT="$ROOT/cadvit_novelty_pilots_cached"
CACHE="$ROOT/cadvit_v2_cache"
read -r -p "Approved free GPU index (0 or 1): " GPU
../.venv/bin/python -m novelty.cadvit_pilot preflight --artifact-root "$ROOT" --output-root "$OUT" --cache-mode on --cache-root "$CACHE" --dataset DDR --ablation p2_p3_probe --micro-batch 1 --epochs 2
# Full cache warm-up only after separate approval (not run in Phase 3B):
../.venv/bin/python -m novelty.cadvit_pilot cache-warmup --artifact-root "$ROOT" --output-root "$OUT" --cache-mode on --cache-root "$CACHE" --warm-split both --warm-limit 11270 --dataset DDR --ablation p2_p3_probe --micro-batch 1 --epochs 2
CUDA_VISIBLE_DEVICES="$GPU" ../.venv/bin/python -m novelty.cadvit_pilot smoke --artifact-root "$ROOT" --output-root "$OUT" --cache-mode on --cache-root "$CACHE" --dataset DDR --ablation p2_p3_probe --micro-batch 1 --epochs 2 --device cuda
# Hanya setelah gate 512×512 GPU dan review ilmiah:
CUDA_VISIBLE_DEVICES="$GPU" ../.venv/bin/python -m novelty.cadvit_pilot pilot --artifact-root "$ROOT" --output-root "$OUT" --cache-mode on --cache-root "$CACHE" --dataset DDR --ablation p2_p3_probe --micro-batch 1 --epochs 2 --device cuda
CUDA_VISIBLE_DEVICES="$GPU" ../.venv/bin/python -m novelty.cadvit_pilot resume --artifact-root "$ROOT" --output-root "$OUT" --cache-mode on --cache-root "$CACHE" --dataset DDR --ablation p2_p3_probe --micro-batch 1 --epochs 2 --device cuda
```

Output: `$OUT/<dataset>/p2_p3_probe_seed42_e2_mb1_eff16_fp32/`. `pilot` menolak direktori yang ada; `resume` memerlukan config, history, best/last, source dan RNG yang kompatibel. Mengubah epoch/microbatch menghasilkan konfigurasi/direktori berbeda. Output harus di luar repository/dataset/arsip/foundation. Estimasi dua checkpoint FP32+AdamW: ~125–150 MiB per arm, bukan ukuran terukur.

## 6. Hasil dan batas ilmiah

| Dataset | Arm | Training | Validation | Plot / confusion matrix |
| --- | --- | --- | --- | --- |
| APTOS2019 | Semua P1–P4 | **NOT RUN** | **NOT RUN** | **NOT AVAILABLE** |
| DDR | Semua P1–P4 | **NOT RUN** | **NOT RUN** | **NOT AVAILABLE** |

Tidak ada hasil atau grafik P1–P4 yang sah untuk diinterpretasikan. Hasil reproduksi historis tidak boleh dipindahkan ke tabel ini.

**Prior art:** P2 terkait [adaptive multiscale weighting DR](https://www.mdpi.com/2079-9292/10/12/1369); [coordinate attention](https://openaccess.thecvf.com/content/CVPR2021/papers/Hou_Coordinate_Attention_for_Efficient_Mobile_Network_Design_CVPR_2021_paper.pdf) dan [target ordinal kumulatif](https://openaccess.thecvf.com/content_ECCVW_2018/papers/11134/Liu_Ordinal_Regression_with_Neuron_Stick-breaking_for_Medical_Diagnosis_ECCVW_2018_paper.pdf) juga sudah dikenal. Mekanisme yang diuji adalah manfaat tambahan **modulasi fitur berbasis prediksi batas grade lokal** setelah ordinary gating dan supervisi ordinal dikendalikan. Audit prior art lebih luas serta konfirmasi independen tetap diperlukan sebelum klaim kontribusi metodologis.

**Keterbatasan:** P1 mungkin menangkap noise; P2 mungkin hanya mereweight cabang lama; P3 probe dapat kurang terkalibrasi; P4 bisa bersaing dengan CE; P3/P4 mengulang target ordinal; BatchNorm bisa mengubah lintasan training walau output awal setara. Selisih parameter 5,28M vs ~7,26M, preprocessing BG10, recovery DDR campuran, resep augmentasi V2 yang belum diverifikasi, dan ketidakpastian split paper membatasi kesetaraan reproduksi. Validation yang juga memilih checkpoint awal bukan konfirmasi independen. Tidak ada bukti pelestarian lesi, kausalitas klinis, atau kesiapan klinis.

**Reproduksibilitas:** sebelum pilot, ulangi preflight APTOS/DDR, catat hash source/manifest/checkpoint, image count, library versions, free disk, dan peak VRAM smoke sintetis 512×512. Bekukan arm/microbatch/epoch sebelum training. Isi bagian hasil hanya dari artefak run yang benar-benar ada. Jangan memakai final test untuk pengembangan. Source: [model](novelty/cadvit_model.py), [loss/transfer](novelty/cadvit_training.py), [runner](novelty/cadvit_pilot.py), [tes model](tests/test_cadvit_novelty.py), [tes runner](tests/test_cadvit_pilot.py), [protokol](docs/CADVIT_NOVELTY_PROTOCOL.md).

## 7. ConvNeXt-Tiny B0–B4: studi penyaringan validasi

Bagian 1–6 tetap khusus CAD-ViT P1–P4 dan berstatus **NOT RUN**. Bagian 7 memuat sepuluh run ConvNeXt-Tiny 20 epoch yang sudah selesai di KC: enam APTOS2019 dan empat DDR. Ini hasil **validasi**, bukan final test, bukan hasil P1–P4, dan bukan bukti novelty.

Navigasi: [protokol](#71-protokol) · [perbandingan](#72-perbandingan-agregat) · [per kelas](#73-per-kelas) · [figur dan 200 epoch](#74-figur-dan-riwayat-epoch) · [interpretasi](#75-interpretasi-ilmiah) · [provenance](#76-provenance).

[Bukti JSON](novelty/convnext_b0_b4/validation_evidence.json) menyimpan **seluruh objek asli** dari 200 epoch, termasuk field yang tidak muat dalam tabel. Tanda “—” berarti tidak tersedia; B0 memang tidak merekam train accuracy/macro-F1.


### 7.1 Protokol

| Arm | Kondisi | Dataset |
| --- | --- | --- |
| B0 | Full fine-tuning vanilla, ordinary CE | APTOS2019, DDR |
| B1 | Augmentasi konservatif training | APTOS2019, DDR |
| B2 | Class-weighted CE tanpa B1 augmentation | APTOS2019, DDR |
| B3 | B1 + sampling grade 3 target 10% per epoch | APTOS2019, DDR |
| B4-Q | B1 + karantina 50 train rows dari 24 grup SHA label-konflik | APTOS2019 |
| B4-R | B1 + kontrol penghapusan 50 train rows dari 27 grup SHA nonkonflik utuh, class-matched | APTOS2019 |

Semua run: seed 42, 20 epoch, official ConvNeXt-Tiny pretrained dan processor 224 px, batch efektif 32, BF16, AdamW, backbone LR 1e-5, head LR 1e-3, weight decay 0,01. Checkpoint dipilih dengan validation macro-F1 tertinggi; tie: validation loss lebih kecil, lalu epoch paling awal. B1: flip horizontal p=0,50, rotasi p=0,50 ±7°, brightness/contrast masing-masing p=0,30 faktor 0,90–1,10. B2 memakai inverse-square-root class weights dari statistik training (lihat [dokumen kesiapan](docs/FOUNDATION_TRAINING_READINESS.md)). B3 mengubah komposisi sampling tanpa mengubah ukuran epoch. B4 tidak memakai sampler B3.

Split beku: APTOS2019 2.930 train/366 validation, manifest SHA-256 `73d1546e3c2fd2bf9e3d0fd10af914cb523d28e4bc8e1f25212af454d4be8854`; DDR 10.018/1.252, `7bc3e3916e012bc46fcd9cd69aa469e0daf3229e1273f1d8ecb6054899bf6eb1`. B4-Q/R masing-masing 2.880 train dan tetap 366 validation. DDR tidak mempunyai grup SHA identik berlabel konflik menurut audit manifest, jadi **B4 DDR tidak dijalankan sesuai desain**. B3/B4 executable source eksternal KC, belum tracked Git, dan tidak diimpor notebook.


### 7.2 Perbandingan agregat

| Dataset | Arm | Best epoch | Accuracy | Macro-F1 | Weighted-F1 | QWK | Grade MAE |
| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: |
| APTOS2019 | B0 | 17 | 0.8142 | 0.6737 | 0.8171 | 0.8892 | 0.2432 |
| APTOS2019 | B1 | 16 | 0.8306 | 0.7011 | 0.8321 | 0.8986 | 0.2213 |
| APTOS2019 | B2 | 7 | 0.7978 | 0.6600 | 0.8028 | 0.8825 | 0.2623 |
| APTOS2019 | B3 | 20 | 0.8279 | 0.7006 | 0.8298 | 0.8999 | 0.2213 |
| APTOS2019 | B4-Q | 11 | 0.8197 | 0.6883 | 0.8233 | 0.8940 | 0.2350 |
| APTOS2019 | B4-R | 9 | 0.8279 | 0.7079 | 0.8311 | 0.8989 | 0.2213 |
| DDR | B0 | 8 | 0.8075 | 0.6198 | 0.7953 | 0.8203 | 0.3123 |
| DDR | B1 | 11 | 0.8267 | 0.6521 | 0.8182 | 0.8435 | 0.2756 |
| DDR | B2 | 5 | 0.7708 | 0.6231 | 0.7748 | 0.8009 | 0.3610 |
| DDR | B3 | 11 | 0.8187 | 0.6510 | 0.8137 | 0.8432 | 0.2819 |

Macro-F1 merata-ratakan F1 lima grade; weighted-F1 memakai support. QWK mempertimbangkan jarak ordinal. Semuanya metrik validasi checkpoint terbaik. Perbandingan bersifat deskriptif, satu seed, dan menggunakan split yang sama untuk memilih checkpoint.


### 7.3 Per kelas

**APTOS2019.** Presisi, recall, F1, dan support asli dari `best.validation.per_grade`.

| Arm | Grade | Precision | Recall | F1 | Support |
| --- | ---: | ---: | ---: | ---: | ---: |
| B0 | 0 | 0.9943 | 0.9667 | 0.9803 | 180 |
| B0 | 1 | 0.5000 | 0.6571 | 0.5679 | 35 |
| B0 | 2 | 0.7600 | 0.7525 | 0.7562 | 101 |
| B0 | 3 | 0.6250 | 0.5000 | 0.5556 | 20 |
| B0 | 4 | 0.5172 | 0.5000 | 0.5085 | 30 |
| B1 | 0 | 0.9943 | 0.9667 | 0.9803 | 180 |
| B1 | 1 | 0.5435 | 0.7143 | 0.6173 | 35 |
| B1 | 2 | 0.7647 | 0.7723 | 0.7685 | 101 |
| B1 | 3 | 0.6429 | 0.4500 | 0.5294 | 20 |
| B1 | 4 | 0.6207 | 0.6000 | 0.6102 | 30 |
| B2 | 0 | 0.9884 | 0.9500 | 0.9688 | 180 |
| B2 | 1 | 0.4565 | 0.6000 | 0.5185 | 35 |
| B2 | 2 | 0.7475 | 0.7327 | 0.7400 | 101 |
| B2 | 3 | 0.5789 | 0.5500 | 0.5641 | 20 |
| B2 | 4 | 0.5172 | 0.5000 | 0.5085 | 30 |
| B3 | 0 | 0.9943 | 0.9667 | 0.9803 | 180 |
| B3 | 1 | 0.5349 | 0.6571 | 0.5897 | 35 |
| B3 | 2 | 0.7500 | 0.7723 | 0.7610 | 101 |
| B3 | 3 | 0.5882 | 0.5000 | 0.5405 | 20 |
| B3 | 4 | 0.6667 | 0.6000 | 0.6316 | 30 |
| B4-Q | 0 | 0.9943 | 0.9722 | 0.9831 | 180 |
| B4-Q | 1 | 0.5349 | 0.6571 | 0.5897 | 35 |
| B4-Q | 2 | 0.7684 | 0.7228 | 0.7449 | 101 |
| B4-Q | 3 | 0.5000 | 0.5500 | 0.5238 | 20 |
| B4-Q | 4 | 0.6000 | 0.6000 | 0.6000 | 30 |
| B4-R | 0 | 0.9943 | 0.9667 | 0.9803 | 180 |
| B4-R | 1 | 0.5122 | 0.6000 | 0.5526 | 35 |
| B4-R | 2 | 0.7700 | 0.7624 | 0.7662 | 101 |
| B4-R | 3 | 0.5600 | 0.7000 | 0.6222 | 20 |
| B4-R | 4 | 0.6800 | 0.5667 | 0.6182 | 30 |

Support grade 1 = 35 dan grade 3 = 20. B1 recall grade 1 0,7143; B4-R recall grade 3 0,7000 tetapi recall grade 1 0,6000.


#### DDR

Presisi, recall, F1, dan support asli dari `best.validation.per_grade`.

| Arm | Grade | Precision | Recall | F1 | Support |
| --- | ---: | ---: | ---: | ---: | ---: |
| B0 | 0 | 0.8779 | 0.9073 | 0.8924 | 626 |
| B0 | 1 | 0.2632 | 0.0794 | 0.1220 | 63 |
| B0 | 2 | 0.7531 | 0.8036 | 0.7775 | 448 |
| B0 | 3 | 0.5172 | 0.6250 | 0.5660 | 24 |
| B0 | 4 | 0.7975 | 0.6923 | 0.7412 | 91 |
| B1 | 0 | 0.8899 | 0.9169 | 0.9032 | 626 |
| B1 | 1 | 0.3143 | 0.1746 | 0.2245 | 63 |
| B1 | 2 | 0.7811 | 0.8281 | 0.8039 | 448 |
| B1 | 3 | 0.7692 | 0.4167 | 0.5405 | 24 |
| B1 | 4 | 0.8214 | 0.7582 | 0.7886 | 91 |
| B2 | 0 | 0.8938 | 0.8339 | 0.8628 | 626 |
| B2 | 1 | 0.2353 | 0.2540 | 0.2443 | 63 |
| B2 | 2 | 0.7286 | 0.7612 | 0.7445 | 448 |
| B2 | 3 | 0.4286 | 0.6250 | 0.5085 | 24 |
| B2 | 4 | 0.7320 | 0.7802 | 0.7553 | 91 |
| B3 | 0 | 0.8813 | 0.9249 | 0.9026 | 626 |
| B3 | 1 | 0.2708 | 0.2063 | 0.2342 | 63 |
| B3 | 2 | 0.8014 | 0.7746 | 0.7877 | 448 |
| B3 | 3 | 0.5455 | 0.5000 | 0.5217 | 24 |
| B3 | 4 | 0.8043 | 0.8132 | 0.8087 | 91 |

Support grade 1 = 63 dan grade 3 = 24. B2 recall grade 1 0,2540, melebihi B1 0,1746 walaupun macro-F1 B2 lebih rendah. B0/B2 recall grade 3 0,6250, B1 0,4167.


### 7.4 Figur dan riwayat epoch

Kurva berikut menampilkan training/validation 20 epoch. Matriks memakai checkpoint terbaik; **baris adalah grade sebenarnya, kolom prediksi**. Delapan SVG B1–B4 disalin byte-for-byte dari run; dua SVG B0 dibuat dari history asli memakai fungsi pelaporan yang sudah ada. JSON mempertahankan per-grade, matriks per epoch, resource, learning rate, dan hitungan langkah bila ada.


#### APTOS2019 — B0: 20 epoch

**Konteks.** Best epoch 17, macro-F1 0.6737; macro-F1 epoch 20 0.6653. Validation loss minimum pada epoch 6 (0.5000); akhir 0.8653.

![Kurva APTOS2019 B0](novelty/convnext_b0_b4/APTOS2019_B0_learning_curves.svg)

**Interpretasi kurva.** Loss minimum dan macro-F1 terbaik adalah dua kriteria berbeda. Kenaikan loss validasi akhir konsisten dengan risiko overfitting, tanpa menetapkan penyebab.

![Matriks kebingungan APTOS2019 B0](novelty/convnext_b0_b4/APTOS2019_B0_confusion_matrix.png)

**Interpretasi matriks.** 298/366 tepat. Tiga sel salah terbesar: 2→1: 15, 1→2: 11, 4→2: 10. Recall grade 1 0.6571 (n=35); grade 3 0.5000 (n=20). Ini deskriptif, bukan diagnosis penyebab klinis.

| Epoch | Train loss | Train acc | Train macro-F1 | Val loss | Val acc | Val macro-F1 | Val weighted-F1 | QWK | MAE |
| ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 1 | 0.8162 | — | — | 0.6244 | 0.7623 | 0.4311 | 0.7146 | 0.7895 | 0.3388 |
| 2 | 0.5419 | — | — | 0.5622 | 0.7760 | 0.4670 | 0.7355 | 0.8164 | 0.3115 |
| 3 | 0.4577 | — | — | 0.5184 | 0.8005 | 0.5771 | 0.7789 | 0.8656 | 0.2623 |
| 4 | 0.4002 | — | — | 0.5019 | 0.8087 | 0.6234 | 0.7995 | 0.8678 | 0.2596 |
| 5 | 0.3549 | — | — | 0.5130 | 0.8197 | 0.6372 | 0.8081 | 0.8813 | 0.2350 |
| 6 | 0.3086 | — | — | 0.5000 | 0.8033 | 0.6069 | 0.7920 | 0.8806 | 0.2486 |
| 7 | 0.2636 | — | — | 0.5103 | 0.8115 | 0.6521 | 0.8096 | 0.8870 | 0.2432 |
| 8 | 0.2214 | — | — | 0.5116 | 0.8169 | 0.6486 | 0.8123 | 0.8885 | 0.2377 |
| 9 | 0.1785 | — | — | 0.5173 | 0.8115 | 0.6599 | 0.8125 | 0.8886 | 0.2486 |
| 10 | 0.1412 | — | — | 0.5415 | 0.8169 | 0.6575 | 0.8138 | 0.8769 | 0.2432 |
| 11 | 0.1104 | — | — | 0.5768 | 0.7978 | 0.6519 | 0.8015 | 0.8784 | 0.2705 |
| 12 | 0.0838 | — | — | 0.6000 | 0.8197 | 0.6649 | 0.8189 | 0.8885 | 0.2377 |
| 13 | 0.0639 | — | — | 0.6420 | 0.8251 | 0.6660 | 0.8216 | 0.8923 | 0.2268 |
| 14 | 0.0522 | — | — | 0.6435 | 0.8060 | 0.6409 | 0.8038 | 0.8715 | 0.2568 |
| 15 | 0.0446 | — | — | 0.6859 | 0.8142 | 0.6616 | 0.8144 | 0.8875 | 0.2432 |
| 16 | 0.0400 | — | — | 0.8161 | 0.8142 | 0.6509 | 0.8119 | 0.8860 | 0.2404 |
| 17 | 0.0375 | — | — | 0.7804 | 0.8142 | 0.6737 | 0.8171 | 0.8892 | 0.2432 |
| 18 | 0.0366 | — | — | 0.7892 | 0.8142 | 0.6545 | 0.8130 | 0.8888 | 0.2432 |
| 19 | 0.0409 | — | — | 0.8236 | 0.8142 | 0.6586 | 0.8129 | 0.8746 | 0.2514 |
| 20 | 0.0416 | — | — | 0.8653 | 0.8169 | 0.6653 | 0.8178 | 0.8947 | 0.2322 |


#### APTOS2019 — B1: 20 epoch

**Konteks.** Best epoch 16, macro-F1 0.7011; macro-F1 epoch 20 0.6862. Validation loss minimum pada epoch 8 (0.4791); akhir 0.6600.

![Kurva APTOS2019 B1](novelty/convnext_b0_b4/APTOS2019_B1_learning_curves.svg)

**Interpretasi kurva.** Loss minimum dan macro-F1 terbaik adalah dua kriteria berbeda. Kenaikan loss validasi akhir konsisten dengan risiko overfitting, tanpa menetapkan penyebab.

![Matriks kebingungan APTOS2019 B1](novelty/convnext_b0_b4/APTOS2019_B1_confusion_matrix.png)

**Interpretasi matriks.** 304/366 tepat. Tiga sel salah terbesar: 2→1: 13, 1→2: 9, 4→2: 8. Recall grade 1 0.7143 (n=35); grade 3 0.4500 (n=20). Ini deskriptif, bukan diagnosis penyebab klinis.

| Epoch | Train loss | Train acc | Train macro-F1 | Val loss | Val acc | Val macro-F1 | Val weighted-F1 | QWK | MAE |
| ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 1 | 0.8192 | 0.7038 | 0.3585 | 0.6266 | 0.7568 | 0.4238 | 0.7097 | 0.7874 | 0.3443 |
| 2 | 0.5500 | 0.7915 | 0.4989 | 0.5668 | 0.7650 | 0.4442 | 0.7233 | 0.8113 | 0.3224 |
| 3 | 0.4755 | 0.8229 | 0.6156 | 0.5201 | 0.7923 | 0.5492 | 0.7670 | 0.8589 | 0.2732 |
| 4 | 0.4317 | 0.8317 | 0.6560 | 0.4952 | 0.8169 | 0.6301 | 0.8056 | 0.8758 | 0.2432 |
| 5 | 0.3973 | 0.8420 | 0.6743 | 0.5146 | 0.8115 | 0.6117 | 0.7971 | 0.8779 | 0.2432 |
| 6 | 0.3641 | 0.8614 | 0.7222 | 0.4900 | 0.8142 | 0.6383 | 0.8052 | 0.8873 | 0.2350 |
| 7 | 0.3327 | 0.8771 | 0.7559 | 0.4968 | 0.8115 | 0.6539 | 0.8091 | 0.8886 | 0.2404 |
| 8 | 0.3034 | 0.8863 | 0.7794 | 0.4791 | 0.8197 | 0.6547 | 0.8140 | 0.8962 | 0.2268 |
| 9 | 0.2740 | 0.8976 | 0.8013 | 0.4880 | 0.8224 | 0.6746 | 0.8227 | 0.8967 | 0.2268 |
| 10 | 0.2366 | 0.9164 | 0.8321 | 0.4915 | 0.8224 | 0.6787 | 0.8229 | 0.9004 | 0.2213 |
| 11 | 0.2116 | 0.9297 | 0.8638 | 0.5116 | 0.8251 | 0.7010 | 0.8286 | 0.9013 | 0.2240 |
| 12 | 0.1889 | 0.9355 | 0.8694 | 0.5198 | 0.8361 | 0.6898 | 0.8331 | 0.9006 | 0.2104 |
| 13 | 0.1637 | 0.9461 | 0.8944 | 0.5485 | 0.8361 | 0.6820 | 0.8312 | 0.8999 | 0.2104 |
| 14 | 0.1407 | 0.9563 | 0.9148 | 0.5206 | 0.8169 | 0.6737 | 0.8167 | 0.8935 | 0.2350 |
| 15 | 0.1243 | 0.9594 | 0.9229 | 0.5542 | 0.8251 | 0.6913 | 0.8265 | 0.8920 | 0.2322 |
| 16 | 0.0999 | 0.9696 | 0.9375 | 0.6049 | 0.8306 | 0.7011 | 0.8321 | 0.8986 | 0.2213 |
| 17 | 0.0872 | 0.9737 | 0.9488 | 0.5878 | 0.8224 | 0.6879 | 0.8235 | 0.8976 | 0.2295 |
| 18 | 0.0729 | 0.9792 | 0.9580 | 0.6050 | 0.8142 | 0.6733 | 0.8156 | 0.8893 | 0.2459 |
| 19 | 0.0664 | 0.9802 | 0.9606 | 0.6292 | 0.8142 | 0.6779 | 0.8151 | 0.8925 | 0.2404 |
| 20 | 0.0629 | 0.9775 | 0.9554 | 0.6600 | 0.8169 | 0.6862 | 0.8193 | 0.8950 | 0.2350 |


#### APTOS2019 — B2: 20 epoch

**Konteks.** Best epoch 7, macro-F1 0.6600; macro-F1 epoch 20 0.6358. Validation loss minimum pada epoch 6 (0.5112); akhir 0.9200.

![Kurva APTOS2019 B2](novelty/convnext_b0_b4/APTOS2019_B2_learning_curves.svg)

**Interpretasi kurva.** Loss minimum dan macro-F1 terbaik adalah dua kriteria berbeda. Kenaikan loss validasi akhir konsisten dengan risiko overfitting, tanpa menetapkan penyebab.

![Matriks kebingungan APTOS2019 B2](novelty/convnext_b0_b4/APTOS2019_B2_confusion_matrix.png)

**Interpretasi matriks.** 292/366 tepat. Tiga sel salah terbesar: 2→1: 14, 1→2: 13, 4→2: 10. Recall grade 1 0.6000 (n=35); grade 3 0.5500 (n=20). Ini deskriptif, bukan diagnosis penyebab klinis.

| Epoch | Train loss | Train acc | Train macro-F1 | Val loss | Val acc | Val macro-F1 | Val weighted-F1 | QWK | MAE |
| ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 1 | 1.0429 | 0.7119 | 0.4309 | 0.6893 | 0.7459 | 0.5576 | 0.7415 | 0.8256 | 0.3333 |
| 2 | 0.7311 | 0.8082 | 0.6361 | 0.5921 | 0.7842 | 0.5764 | 0.7719 | 0.8503 | 0.2842 |
| 3 | 0.6258 | 0.8304 | 0.6887 | 0.5396 | 0.7869 | 0.5991 | 0.7811 | 0.8731 | 0.2760 |
| 4 | 0.5487 | 0.8485 | 0.7300 | 0.5318 | 0.7732 | 0.6140 | 0.7785 | 0.8539 | 0.3169 |
| 5 | 0.4880 | 0.8676 | 0.7649 | 0.5267 | 0.8033 | 0.6400 | 0.8028 | 0.8776 | 0.2541 |
| 6 | 0.4188 | 0.8857 | 0.8010 | 0.5112 | 0.8087 | 0.6529 | 0.8071 | 0.8860 | 0.2459 |
| 7 | 0.3552 | 0.9068 | 0.8355 | 0.5325 | 0.7978 | 0.6600 | 0.8028 | 0.8825 | 0.2623 |
| 8 | 0.2917 | 0.9270 | 0.8761 | 0.5300 | 0.8060 | 0.6513 | 0.8078 | 0.8765 | 0.2650 |
| 9 | 0.2349 | 0.9461 | 0.9081 | 0.5454 | 0.7978 | 0.6505 | 0.8022 | 0.8779 | 0.2705 |
| 10 | 0.1851 | 0.9608 | 0.9283 | 0.5590 | 0.8060 | 0.6428 | 0.8045 | 0.8702 | 0.2623 |
| 11 | 0.1436 | 0.9672 | 0.9422 | 0.6023 | 0.8005 | 0.6582 | 0.8052 | 0.8832 | 0.2650 |
| 12 | 0.1102 | 0.9765 | 0.9575 | 0.6373 | 0.8033 | 0.6504 | 0.8051 | 0.8796 | 0.2596 |
| 13 | 0.0866 | 0.9812 | 0.9638 | 0.6661 | 0.8142 | 0.6514 | 0.8129 | 0.8942 | 0.2350 |
| 14 | 0.0698 | 0.9843 | 0.9698 | 0.6689 | 0.8142 | 0.6590 | 0.8142 | 0.8812 | 0.2432 |
| 15 | 0.0629 | 0.9857 | 0.9695 | 0.7197 | 0.8087 | 0.6445 | 0.8082 | 0.8739 | 0.2568 |
| 16 | 0.0554 | 0.9860 | 0.9715 | 0.8357 | 0.8087 | 0.6382 | 0.8075 | 0.8924 | 0.2377 |
| 17 | 0.0521 | 0.9867 | 0.9731 | 0.8127 | 0.8033 | 0.6520 | 0.8068 | 0.8869 | 0.2541 |
| 18 | 0.0490 | 0.9881 | 0.9761 | 0.8347 | 0.8005 | 0.6401 | 0.8015 | 0.8816 | 0.2596 |
| 19 | 0.0576 | 0.9870 | 0.9727 | 0.8689 | 0.8115 | 0.6465 | 0.8101 | 0.8746 | 0.2541 |
| 20 | 0.0543 | 0.9860 | 0.9718 | 0.9200 | 0.8005 | 0.6358 | 0.8013 | 0.8765 | 0.2568 |


#### APTOS2019 — B3: 20 epoch

**Konteks.** Best epoch 20, macro-F1 0.7006; macro-F1 epoch 20 0.7006. Validation loss minimum pada epoch 8 (0.4900); akhir 0.7180.

![Kurva APTOS2019 B3](novelty/convnext_b0_b4/APTOS2019_B3_learning_curves.svg)

**Interpretasi kurva.** Loss minimum dan macro-F1 terbaik adalah dua kriteria berbeda. Kenaikan loss validasi akhir konsisten dengan risiko overfitting, tanpa menetapkan penyebab.

![Matriks kebingungan APTOS2019 B3](novelty/convnext_b0_b4/APTOS2019_B3_confusion_matrix.png)

**Interpretasi matriks.** 303/366 tepat. Tiga sel salah terbesar: 2→1: 12, 1→2: 11, 4→2: 8. Recall grade 1 0.6571 (n=35); grade 3 0.5000 (n=20). Ini deskriptif, bukan diagnosis penyebab klinis.

| Epoch | Train loss | Train acc | Train macro-F1 | Val loss | Val acc | Val macro-F1 | Val weighted-F1 | QWK | MAE |
| ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 1 | 0.8680 | 0.6758 | 0.3861 | 0.6355 | 0.7541 | 0.4946 | 0.7266 | 0.8280 | 0.3251 |
| 2 | 0.5849 | 0.7853 | 0.5925 | 0.5775 | 0.7678 | 0.4972 | 0.7355 | 0.8358 | 0.3060 |
| 3 | 0.5023 | 0.8143 | 0.6563 | 0.5246 | 0.7787 | 0.5377 | 0.7557 | 0.8511 | 0.2896 |
| 4 | 0.4573 | 0.8249 | 0.6883 | 0.5077 | 0.8060 | 0.6384 | 0.7998 | 0.8756 | 0.2514 |
| 5 | 0.4176 | 0.8324 | 0.7030 | 0.4979 | 0.7978 | 0.6091 | 0.7903 | 0.8799 | 0.2514 |
| 6 | 0.3796 | 0.8560 | 0.7426 | 0.4917 | 0.8142 | 0.6418 | 0.8058 | 0.8880 | 0.2350 |
| 7 | 0.3462 | 0.8720 | 0.7795 | 0.4979 | 0.8115 | 0.6510 | 0.8089 | 0.8934 | 0.2322 |
| 8 | 0.3113 | 0.8881 | 0.8095 | 0.4900 | 0.8060 | 0.6220 | 0.7989 | 0.8916 | 0.2404 |
| 9 | 0.2788 | 0.8980 | 0.8232 | 0.4943 | 0.8251 | 0.6922 | 0.8276 | 0.9022 | 0.2186 |
| 10 | 0.2348 | 0.9188 | 0.8618 | 0.5034 | 0.8306 | 0.6995 | 0.8329 | 0.9094 | 0.2077 |
| 11 | 0.2078 | 0.9287 | 0.8803 | 0.5207 | 0.8251 | 0.6934 | 0.8294 | 0.9054 | 0.2186 |
| 12 | 0.1875 | 0.9362 | 0.8932 | 0.5201 | 0.8087 | 0.6576 | 0.8092 | 0.8911 | 0.2404 |
| 13 | 0.1604 | 0.9478 | 0.9144 | 0.5574 | 0.8142 | 0.6351 | 0.8101 | 0.8967 | 0.2322 |
| 14 | 0.1341 | 0.9594 | 0.9332 | 0.5380 | 0.8279 | 0.6915 | 0.8275 | 0.9036 | 0.2158 |
| 15 | 0.1186 | 0.9621 | 0.9386 | 0.5712 | 0.8224 | 0.6773 | 0.8231 | 0.8945 | 0.2322 |
| 16 | 0.0977 | 0.9703 | 0.9520 | 0.6312 | 0.8142 | 0.6778 | 0.8182 | 0.8953 | 0.2377 |
| 17 | 0.0865 | 0.9710 | 0.9519 | 0.6146 | 0.8197 | 0.6866 | 0.8227 | 0.8973 | 0.2322 |
| 18 | 0.0755 | 0.9775 | 0.9627 | 0.6188 | 0.8169 | 0.6572 | 0.8159 | 0.8925 | 0.2404 |
| 19 | 0.0635 | 0.9805 | 0.9671 | 0.6520 | 0.8115 | 0.6723 | 0.8122 | 0.8938 | 0.2404 |
| 20 | 0.0584 | 0.9816 | 0.9688 | 0.7180 | 0.8279 | 0.7006 | 0.8298 | 0.8999 | 0.2213 |


#### APTOS2019 — B4-Q: 20 epoch

**Konteks.** Best epoch 11, macro-F1 0.6883; macro-F1 epoch 20 0.6829. Validation loss minimum pada epoch 5 (0.4930); akhir 0.7965.

![Kurva APTOS2019 B4-Q](novelty/convnext_b0_b4/APTOS2019_B4_Q_learning_curves.svg)

**Interpretasi kurva.** Loss minimum dan macro-F1 terbaik adalah dua kriteria berbeda. Kenaikan loss validasi akhir konsisten dengan risiko overfitting, tanpa menetapkan penyebab.

![Matriks kebingungan APTOS2019 B4-Q](novelty/convnext_b0_b4/APTOS2019_B4_Q_confusion_matrix.png)

**Interpretasi matriks.** 300/366 tepat. Tiga sel salah terbesar: 2→1: 13, 1→2: 10, 2→3: 9. Recall grade 1 0.6571 (n=35); grade 3 0.5500 (n=20). Ini deskriptif, bukan diagnosis penyebab klinis.

| Epoch | Train loss | Train acc | Train macro-F1 | Val loss | Val acc | Val macro-F1 | Val weighted-F1 | QWK | MAE |
| ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 1 | 0.7918 | 0.7191 | 0.3700 | 0.6164 | 0.7514 | 0.3860 | 0.6941 | 0.7932 | 0.3470 |
| 2 | 0.5270 | 0.7976 | 0.4944 | 0.5578 | 0.7896 | 0.5461 | 0.7626 | 0.8451 | 0.2842 |
| 3 | 0.4634 | 0.8215 | 0.5825 | 0.5236 | 0.7923 | 0.5668 | 0.7720 | 0.8589 | 0.2732 |
| 4 | 0.4154 | 0.8451 | 0.6765 | 0.5082 | 0.8060 | 0.6025 | 0.7901 | 0.8782 | 0.2541 |
| 5 | 0.3806 | 0.8556 | 0.6947 | 0.4930 | 0.8033 | 0.6248 | 0.7975 | 0.8840 | 0.2486 |
| 6 | 0.3476 | 0.8670 | 0.7305 | 0.4934 | 0.8060 | 0.6203 | 0.7971 | 0.8798 | 0.2514 |
| 7 | 0.3149 | 0.8844 | 0.7629 | 0.5002 | 0.8060 | 0.6237 | 0.7992 | 0.8824 | 0.2486 |
| 8 | 0.2867 | 0.8938 | 0.7880 | 0.5193 | 0.8087 | 0.6754 | 0.8135 | 0.8822 | 0.2623 |
| 9 | 0.2516 | 0.9076 | 0.8189 | 0.5075 | 0.8224 | 0.6811 | 0.8230 | 0.8976 | 0.2240 |
| 10 | 0.2127 | 0.9236 | 0.8510 | 0.5186 | 0.8197 | 0.6639 | 0.8173 | 0.8940 | 0.2268 |
| 11 | 0.1867 | 0.9389 | 0.8794 | 0.5429 | 0.8197 | 0.6883 | 0.8233 | 0.8940 | 0.2350 |
| 12 | 0.1603 | 0.9455 | 0.8923 | 0.5473 | 0.8197 | 0.6695 | 0.8202 | 0.8985 | 0.2295 |
| 13 | 0.1379 | 0.9549 | 0.9114 | 0.5694 | 0.8169 | 0.6613 | 0.8170 | 0.8921 | 0.2404 |
| 14 | 0.1141 | 0.9656 | 0.9311 | 0.5586 | 0.8197 | 0.6648 | 0.8195 | 0.9022 | 0.2268 |
| 15 | 0.0909 | 0.9736 | 0.9469 | 0.5866 | 0.8142 | 0.6674 | 0.8154 | 0.8997 | 0.2322 |
| 16 | 0.0701 | 0.9837 | 0.9671 | 0.6653 | 0.8197 | 0.6857 | 0.8225 | 0.9015 | 0.2240 |
| 17 | 0.0600 | 0.9854 | 0.9691 | 0.6815 | 0.8060 | 0.6553 | 0.8081 | 0.8969 | 0.2404 |
| 18 | 0.0504 | 0.9882 | 0.9755 | 0.7159 | 0.8033 | 0.6516 | 0.8062 | 0.8949 | 0.2459 |
| 19 | 0.0348 | 0.9917 | 0.9815 | 0.7389 | 0.8197 | 0.6803 | 0.8215 | 0.9002 | 0.2268 |
| 20 | 0.0283 | 0.9917 | 0.9809 | 0.7965 | 0.8197 | 0.6829 | 0.8218 | 0.8984 | 0.2295 |


#### APTOS2019 — B4-R: 20 epoch

**Konteks.** Best epoch 9, macro-F1 0.7079; macro-F1 epoch 20 0.6771. Validation loss minimum pada epoch 6 (0.4726); akhir 0.6947.

![Kurva APTOS2019 B4-R](novelty/convnext_b0_b4/APTOS2019_B4_R_learning_curves.svg)

**Interpretasi kurva.** Loss minimum dan macro-F1 terbaik adalah dua kriteria berbeda. Kenaikan loss validasi akhir konsisten dengan risiko overfitting, tanpa menetapkan penyebab.

![Matriks kebingungan APTOS2019 B4-R](novelty/convnext_b0_b4/APTOS2019_B4_R_confusion_matrix.png)

**Interpretasi matriks.** 303/366 tepat. Tiga sel salah terbesar: 2→1: 12, 1→2: 12, 4→2: 9. Recall grade 1 0.6000 (n=35); grade 3 0.7000 (n=20). Ini deskriptif, bukan diagnosis penyebab klinis.

| Epoch | Train loss | Train acc | Train macro-F1 | Val loss | Val acc | Val macro-F1 | Val weighted-F1 | QWK | MAE |
| ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 1 | 0.8111 | 0.7014 | 0.3495 | 0.6195 | 0.7623 | 0.4312 | 0.7119 | 0.7668 | 0.3525 |
| 2 | 0.5363 | 0.7951 | 0.4857 | 0.5661 | 0.7842 | 0.5164 | 0.7496 | 0.8215 | 0.3005 |
| 3 | 0.4691 | 0.8184 | 0.5860 | 0.5161 | 0.7978 | 0.5612 | 0.7748 | 0.8606 | 0.2678 |
| 4 | 0.4252 | 0.8368 | 0.6539 | 0.4906 | 0.8142 | 0.6484 | 0.8107 | 0.8713 | 0.2568 |
| 5 | 0.3850 | 0.8521 | 0.6971 | 0.4800 | 0.8169 | 0.6366 | 0.8074 | 0.8845 | 0.2377 |
| 6 | 0.3546 | 0.8663 | 0.7229 | 0.4726 | 0.8197 | 0.6474 | 0.8126 | 0.8859 | 0.2350 |
| 7 | 0.3214 | 0.8792 | 0.7553 | 0.4872 | 0.8169 | 0.6439 | 0.8096 | 0.8894 | 0.2322 |
| 8 | 0.2907 | 0.8899 | 0.7778 | 0.4777 | 0.8169 | 0.6721 | 0.8184 | 0.8858 | 0.2459 |
| 9 | 0.2633 | 0.9042 | 0.8096 | 0.4875 | 0.8279 | 0.7079 | 0.8311 | 0.8989 | 0.2213 |
| 10 | 0.2315 | 0.9201 | 0.8428 | 0.4847 | 0.8333 | 0.6920 | 0.8337 | 0.8984 | 0.2186 |
| 11 | 0.1975 | 0.9340 | 0.8706 | 0.5061 | 0.8306 | 0.7041 | 0.8332 | 0.9016 | 0.2158 |
| 12 | 0.1792 | 0.9378 | 0.8768 | 0.5167 | 0.8251 | 0.6665 | 0.8247 | 0.8966 | 0.2268 |
| 13 | 0.1566 | 0.9483 | 0.8984 | 0.5518 | 0.8333 | 0.7016 | 0.8340 | 0.9056 | 0.2077 |
| 14 | 0.1316 | 0.9559 | 0.9114 | 0.5219 | 0.8224 | 0.6794 | 0.8240 | 0.8980 | 0.2268 |
| 15 | 0.1073 | 0.9674 | 0.9337 | 0.5808 | 0.8279 | 0.6963 | 0.8294 | 0.9011 | 0.2186 |
| 16 | 0.0909 | 0.9736 | 0.9485 | 0.5881 | 0.8251 | 0.6847 | 0.8268 | 0.8982 | 0.2268 |
| 17 | 0.0882 | 0.9733 | 0.9469 | 0.6424 | 0.8169 | 0.6655 | 0.8196 | 0.8977 | 0.2377 |
| 18 | 0.0716 | 0.9764 | 0.9520 | 0.6220 | 0.8115 | 0.6730 | 0.8157 | 0.8836 | 0.2541 |
| 19 | 0.0568 | 0.9816 | 0.9619 | 0.6462 | 0.8142 | 0.6816 | 0.8185 | 0.8888 | 0.2486 |
| 20 | 0.0603 | 0.9802 | 0.9596 | 0.6947 | 0.8197 | 0.6771 | 0.8222 | 0.8975 | 0.2322 |


#### DDR — B0: 20 epoch

**Konteks.** Best epoch 8, macro-F1 0.6198; macro-F1 epoch 20 0.5915. Validation loss minimum pada epoch 4 (0.5726); akhir 1.7250.

![Kurva DDR B0](novelty/convnext_b0_b4/DDR_B0_learning_curves.svg)

**Interpretasi kurva.** Loss minimum dan macro-F1 terbaik adalah dua kriteria berbeda. Kenaikan loss validasi akhir konsisten dengan risiko overfitting, tanpa menetapkan penyebab.

![Matriks kebingungan DDR B0](novelty/convnext_b0_b4/DDR_B0_confusion_matrix.png)

**Interpretasi matriks.** 1011/1252 tepat. Tiga sel salah terbesar: 2→0: 57, 0→2: 51, 1→2: 37. Recall grade 1 0.0794 (n=63); grade 3 0.6250 (n=24). Ini deskriptif, bukan diagnosis penyebab klinis.

| Epoch | Train loss | Train acc | Train macro-F1 | Val loss | Val acc | Val macro-F1 | Val weighted-F1 | QWK | MAE |
| ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 1 | 0.7506 | — | — | 0.6397 | 0.7660 | 0.4489 | 0.7359 | 0.7413 | 0.4042 |
| 2 | 0.5605 | — | — | 0.5954 | 0.7899 | 0.5232 | 0.7627 | 0.7796 | 0.3594 |
| 3 | 0.4788 | — | — | 0.5763 | 0.7899 | 0.5405 | 0.7664 | 0.7864 | 0.3586 |
| 4 | 0.4108 | — | — | 0.5726 | 0.7987 | 0.5604 | 0.7770 | 0.7958 | 0.3442 |
| 5 | 0.3392 | — | — | 0.6073 | 0.7955 | 0.5999 | 0.7820 | 0.8010 | 0.3403 |
| 6 | 0.2803 | — | — | 0.6365 | 0.7963 | 0.5796 | 0.7848 | 0.7993 | 0.3347 |
| 7 | 0.2185 | — | — | 0.6812 | 0.8083 | 0.5842 | 0.7899 | 0.8080 | 0.3211 |
| 8 | 0.1559 | — | — | 0.7315 | 0.8075 | 0.6198 | 0.7953 | 0.8203 | 0.3123 |
| 9 | 0.1035 | — | — | 0.8453 | 0.8011 | 0.6009 | 0.7898 | 0.8102 | 0.3219 |
| 10 | 0.0605 | — | — | 0.9887 | 0.8011 | 0.5987 | 0.7893 | 0.8079 | 0.3251 |
| 11 | 0.0297 | — | — | 1.1190 | 0.8059 | 0.6160 | 0.7990 | 0.8338 | 0.3011 |
| 12 | 0.0116 | — | — | 1.2594 | 0.7995 | 0.6113 | 0.7938 | 0.8250 | 0.3147 |
| 13 | 0.0032 | — | — | 1.3962 | 0.7907 | 0.5917 | 0.7827 | 0.8137 | 0.3315 |
| 14 | 0.0012 | — | — | 1.5244 | 0.7835 | 0.5845 | 0.7796 | 0.8127 | 0.3347 |
| 15 | 0.0004 | — | — | 1.5653 | 0.7923 | 0.5948 | 0.7866 | 0.8146 | 0.3259 |
| 16 | 0.0005 | — | — | 1.6156 | 0.7931 | 0.5912 | 0.7851 | 0.8149 | 0.3267 |
| 17 | 0.0009 | — | — | 1.6555 | 0.7923 | 0.5869 | 0.7833 | 0.8105 | 0.3307 |
| 18 | 0.0002 | — | — | 1.6759 | 0.7939 | 0.5889 | 0.7864 | 0.8170 | 0.3227 |
| 19 | 0.0008 | — | — | 1.6993 | 0.7979 | 0.5913 | 0.7888 | 0.8186 | 0.3187 |
| 20 | 0.0007 | — | — | 1.7250 | 0.7947 | 0.5915 | 0.7873 | 0.8192 | 0.3211 |


#### DDR — B1: 20 epoch

**Konteks.** Best epoch 11, macro-F1 0.6521; macro-F1 epoch 20 0.6439. Validation loss minimum pada epoch 8 (0.5412); akhir 1.0762.

![Kurva DDR B1](novelty/convnext_b0_b4/DDR_B1_learning_curves.svg)

**Interpretasi kurva.** Loss minimum dan macro-F1 terbaik adalah dua kriteria berbeda. Kenaikan loss validasi akhir konsisten dengan risiko overfitting, tanpa menetapkan penyebab.

![Matriks kebingungan DDR B1](novelty/convnext_b0_b4/DDR_B1_confusion_matrix.png)

**Interpretasi matriks.** 1035/1252 tepat. Tiga sel salah terbesar: 2→0: 51, 0→2: 40, 1→2: 31. Recall grade 1 0.1746 (n=63); grade 3 0.4167 (n=24). Ini deskriptif, bukan diagnosis penyebab klinis.

| Epoch | Train loss | Train acc | Train macro-F1 | Val loss | Val acc | Val macro-F1 | Val weighted-F1 | QWK | MAE |
| ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 1 | 0.7510 | 0.7175 | 0.3966 | 0.6418 | 0.7644 | 0.4512 | 0.7341 | 0.7385 | 0.4089 |
| 2 | 0.5754 | 0.7908 | 0.4908 | 0.5913 | 0.7835 | 0.5227 | 0.7569 | 0.7725 | 0.3730 |
| 3 | 0.5105 | 0.8164 | 0.5514 | 0.5616 | 0.7955 | 0.5641 | 0.7729 | 0.7913 | 0.3506 |
| 4 | 0.4601 | 0.8327 | 0.5895 | 0.5514 | 0.8019 | 0.5714 | 0.7782 | 0.8071 | 0.3355 |
| 5 | 0.4111 | 0.8493 | 0.6196 | 0.5690 | 0.7987 | 0.6022 | 0.7843 | 0.8037 | 0.3363 |
| 6 | 0.3698 | 0.8672 | 0.6583 | 0.5802 | 0.8011 | 0.5987 | 0.7891 | 0.8081 | 0.3275 |
| 7 | 0.3360 | 0.8810 | 0.6902 | 0.5452 | 0.8275 | 0.5890 | 0.8058 | 0.8355 | 0.2819 |
| 8 | 0.3019 | 0.8917 | 0.7197 | 0.5412 | 0.8251 | 0.6212 | 0.8095 | 0.8347 | 0.2843 |
| 9 | 0.2568 | 0.9086 | 0.7644 | 0.5575 | 0.8259 | 0.6300 | 0.8104 | 0.8356 | 0.2827 |
| 10 | 0.2313 | 0.9157 | 0.7849 | 0.6083 | 0.8291 | 0.6113 | 0.8075 | 0.8348 | 0.2812 |
| 11 | 0.1958 | 0.9325 | 0.8348 | 0.5909 | 0.8267 | 0.6521 | 0.8182 | 0.8435 | 0.2756 |
| 12 | 0.1690 | 0.9397 | 0.8506 | 0.6528 | 0.8219 | 0.6475 | 0.8132 | 0.8311 | 0.2883 |
| 13 | 0.1440 | 0.9491 | 0.8816 | 0.6868 | 0.8291 | 0.6395 | 0.8170 | 0.8470 | 0.2692 |
| 14 | 0.1169 | 0.9607 | 0.9064 | 0.7131 | 0.8179 | 0.6506 | 0.8137 | 0.8447 | 0.2788 |
| 15 | 0.0982 | 0.9674 | 0.9224 | 0.7764 | 0.8227 | 0.6424 | 0.8143 | 0.8357 | 0.2819 |
| 16 | 0.0810 | 0.9737 | 0.9415 | 0.8040 | 0.8083 | 0.6361 | 0.8073 | 0.8412 | 0.2891 |
| 17 | 0.0638 | 0.9791 | 0.9551 | 0.8543 | 0.8235 | 0.6411 | 0.8171 | 0.8533 | 0.2692 |
| 18 | 0.0532 | 0.9831 | 0.9638 | 0.8903 | 0.8275 | 0.6508 | 0.8202 | 0.8580 | 0.2636 |
| 19 | 0.0373 | 0.9883 | 0.9769 | 1.0176 | 0.8187 | 0.6431 | 0.8102 | 0.8399 | 0.2867 |
| 20 | 0.0367 | 0.9877 | 0.9792 | 1.0762 | 0.8147 | 0.6439 | 0.8088 | 0.8348 | 0.2939 |


#### DDR — B2: 20 epoch

**Konteks.** Best epoch 5, macro-F1 0.6231; macro-F1 epoch 20 0.6091. Validation loss minimum pada epoch 4 (0.6110); akhir 1.7015.

![Kurva DDR B2](novelty/convnext_b0_b4/DDR_B2_learning_curves.svg)

**Interpretasi kurva.** Loss minimum dan macro-F1 terbaik adalah dua kriteria berbeda. Kenaikan loss validasi akhir konsisten dengan risiko overfitting, tanpa menetapkan penyebab.

![Matriks kebingungan DDR B2](novelty/convnext_b0_b4/DDR_B2_confusion_matrix.png)

**Interpretasi matriks.** 965/1252 tepat. Tiga sel salah terbesar: 0→2: 73, 2→0: 48, 1→2: 32. Recall grade 1 0.2540 (n=63); grade 3 0.6250 (n=24). Ini deskriptif, bukan diagnosis penyebab klinis.

| Epoch | Train loss | Train acc | Train macro-F1 | Val loss | Val acc | Val macro-F1 | Val weighted-F1 | QWK | MAE |
| ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 1 | 0.9840 | 0.7098 | 0.4406 | 0.6855 | 0.7644 | 0.5454 | 0.7488 | 0.7580 | 0.4002 |
| 2 | 0.7460 | 0.7784 | 0.5856 | 0.6218 | 0.7772 | 0.5893 | 0.7613 | 0.7780 | 0.3786 |
| 3 | 0.6428 | 0.8093 | 0.6421 | 0.6381 | 0.7692 | 0.5933 | 0.7609 | 0.7934 | 0.3770 |
| 4 | 0.5502 | 0.8338 | 0.6924 | 0.6110 | 0.7835 | 0.6132 | 0.7714 | 0.7939 | 0.3650 |
| 5 | 0.4527 | 0.8625 | 0.7499 | 0.6585 | 0.7708 | 0.6231 | 0.7748 | 0.8009 | 0.3610 |
| 6 | 0.3688 | 0.8910 | 0.8091 | 0.7169 | 0.7492 | 0.6074 | 0.7619 | 0.7882 | 0.3794 |
| 7 | 0.2828 | 0.9192 | 0.8638 | 0.6590 | 0.7899 | 0.6110 | 0.7794 | 0.7880 | 0.3538 |
| 8 | 0.1959 | 0.9470 | 0.9123 | 0.7033 | 0.7963 | 0.6223 | 0.7887 | 0.8011 | 0.3355 |
| 9 | 0.1247 | 0.9694 | 0.9518 | 0.8185 | 0.7812 | 0.6001 | 0.7761 | 0.7887 | 0.3570 |
| 10 | 0.0702 | 0.9848 | 0.9757 | 0.9150 | 0.7923 | 0.6090 | 0.7852 | 0.8033 | 0.3347 |
| 11 | 0.0329 | 0.9939 | 0.9901 | 1.0557 | 0.7851 | 0.6059 | 0.7836 | 0.8136 | 0.3331 |
| 12 | 0.0130 | 0.9972 | 0.9962 | 1.2307 | 0.7891 | 0.5836 | 0.7819 | 0.8123 | 0.3323 |
| 13 | 0.0041 | 0.9994 | 0.9994 | 1.3749 | 0.7987 | 0.6040 | 0.7883 | 0.8137 | 0.3259 |
| 14 | 0.0016 | 0.9995 | 0.9994 | 1.4990 | 0.7955 | 0.5943 | 0.7869 | 0.8149 | 0.3267 |
| 15 | 0.0006 | 0.9999 | 0.9998 | 1.5648 | 0.8051 | 0.6142 | 0.7945 | 0.8206 | 0.3147 |
| 16 | 0.0006 | 0.9999 | 0.9998 | 1.5850 | 0.8035 | 0.6150 | 0.7948 | 0.8249 | 0.3123 |
| 17 | 0.0012 | 0.9999 | 0.9998 | 1.6259 | 0.8011 | 0.6113 | 0.7915 | 0.8208 | 0.3179 |
| 18 | 0.0003 | 0.9999 | 0.9998 | 1.6483 | 0.7963 | 0.6050 | 0.7872 | 0.8161 | 0.3259 |
| 19 | 0.0008 | 0.9999 | 0.9998 | 1.6890 | 0.7979 | 0.5911 | 0.7872 | 0.8189 | 0.3219 |
| 20 | 0.0014 | 0.9998 | 0.9995 | 1.7015 | 0.7995 | 0.6091 | 0.7903 | 0.8214 | 0.3187 |


#### DDR — B3: 20 epoch

**Konteks.** Best epoch 11, macro-F1 0.6510; macro-F1 epoch 20 0.6290. Validation loss minimum pada epoch 5 (0.5807); akhir 1.1244.

![Kurva DDR B3](novelty/convnext_b0_b4/DDR_B3_learning_curves.svg)

**Interpretasi kurva.** Loss minimum dan macro-F1 terbaik adalah dua kriteria berbeda. Kenaikan loss validasi akhir konsisten dengan risiko overfitting, tanpa menetapkan penyebab.

![Matriks kebingungan DDR B3](novelty/convnext_b0_b4/DDR_B3_confusion_matrix.png)

**Interpretasi matriks.** 1025/1252 tepat. Tiga sel salah terbesar: 2→0: 56, 0→2: 32, 1→2: 30. Recall grade 1 0.2063 (n=63); grade 3 0.5000 (n=24). Ini deskriptif, bukan diagnosis penyebab klinis.

| Epoch | Train loss | Train acc | Train macro-F1 | Val loss | Val acc | Val macro-F1 | Val weighted-F1 | QWK | MAE |
| ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 1 | 0.8085 | 0.6887 | 0.4845 | 0.6670 | 0.7644 | 0.5071 | 0.7393 | 0.7577 | 0.3930 |
| 2 | 0.6023 | 0.7751 | 0.6012 | 0.6090 | 0.7764 | 0.5524 | 0.7551 | 0.7811 | 0.3754 |
| 3 | 0.5172 | 0.8095 | 0.6467 | 0.5956 | 0.7772 | 0.5626 | 0.7589 | 0.7830 | 0.3762 |
| 4 | 0.4525 | 0.8350 | 0.6860 | 0.5817 | 0.7955 | 0.5791 | 0.7727 | 0.8004 | 0.3450 |
| 5 | 0.3942 | 0.8604 | 0.7166 | 0.5807 | 0.7979 | 0.5930 | 0.7798 | 0.8035 | 0.3371 |
| 6 | 0.3464 | 0.8754 | 0.7466 | 0.5935 | 0.8027 | 0.5954 | 0.7886 | 0.8105 | 0.3235 |
| 7 | 0.3137 | 0.8900 | 0.7717 | 0.5850 | 0.8179 | 0.5981 | 0.7965 | 0.8196 | 0.3011 |
| 8 | 0.2777 | 0.9021 | 0.7982 | 0.5842 | 0.8227 | 0.6262 | 0.8076 | 0.8329 | 0.2875 |
| 9 | 0.2354 | 0.9171 | 0.8263 | 0.5955 | 0.8139 | 0.6239 | 0.8019 | 0.8221 | 0.3019 |
| 10 | 0.2144 | 0.9239 | 0.8477 | 0.6411 | 0.8187 | 0.5880 | 0.7975 | 0.8320 | 0.2923 |
| 11 | 0.1815 | 0.9360 | 0.8725 | 0.6397 | 0.8187 | 0.6510 | 0.8137 | 0.8432 | 0.2819 |
| 12 | 0.1523 | 0.9475 | 0.8962 | 0.6940 | 0.8171 | 0.6290 | 0.8076 | 0.8279 | 0.2955 |
| 13 | 0.1346 | 0.9537 | 0.9098 | 0.7398 | 0.8203 | 0.6322 | 0.8066 | 0.8323 | 0.2891 |
| 14 | 0.1129 | 0.9616 | 0.9266 | 0.7423 | 0.8235 | 0.6335 | 0.8144 | 0.8433 | 0.2780 |
| 15 | 0.0954 | 0.9693 | 0.9429 | 0.8007 | 0.8115 | 0.6232 | 0.8032 | 0.8222 | 0.3027 |
| 16 | 0.0784 | 0.9745 | 0.9542 | 0.8353 | 0.8123 | 0.6455 | 0.8086 | 0.8335 | 0.2915 |
| 17 | 0.0699 | 0.9762 | 0.9565 | 0.8883 | 0.8203 | 0.6307 | 0.8118 | 0.8417 | 0.2788 |
| 18 | 0.0515 | 0.9846 | 0.9726 | 1.0428 | 0.8027 | 0.6084 | 0.7918 | 0.8296 | 0.3091 |
| 19 | 0.0420 | 0.9872 | 0.9776 | 1.0403 | 0.8115 | 0.6308 | 0.8053 | 0.8400 | 0.2907 |
| 20 | 0.0356 | 0.9887 | 0.9825 | 1.1244 | 0.8011 | 0.6290 | 0.7961 | 0.8251 | 0.3139 |


### 7.5 Interpretasi ilmiah

Macro-F1 **tidak sama dengan recall minoritas**. APTOS B4-R macro-F1 0,7079 melebihi B1 0,7011, tetapi recall grade 1 turun dari 0,7143 ke 0,6000. DDR B1 macro-F1 0,6521 melebihi B2 0,6231, tetapi recall grade 1 lebih rendah (0,1746 vs 0,2540). Grade 3 hanya 20 APTOS atau 24 DDR; beberapa prediksi mengubah recall cukup besar. Laporkan per kelas bersama agregat.

B4-R mengungguli B4-Q pada macro-F1 APTOS satu run ini. Namun 24 vs 27 grup yang dikeluarkan, satu seed, dan pemilihan epoch lewat validasi membatasi kesimpulan kausal. Sembilan dari sepuluh best macro-F1 muncul sebelum epoch 20; APTOS B3 pada epoch 20. Seluruh validation loss minimum sebelum epoch terakhir. Batas 50 epoch adalah **rencana**, tanpa hasil 50/100 epoch. Latensi CPU, ONNX, dan kuantisasi juga **belum diukur**. Tidak ada final test ConvNeXt, konfirmasi lintas seed, atau validasi klinis di bagian ini. Hasil ini bukan bukti novelty; lihat [prior art dan rencana benchmark](docs/FOUNDATION_TRAINING_READINESS.md).


### 7.6 Provenance

[Bukti terstruktur](novelty/convnext_b0_b4/validation_evidence.json) menyimpan lokasi relatif sumber KC, SHA-256 `run.json`, `history.json`, `summary.json`, `best.pt`, snapshot processor/weights, konfigurasi relevan, 200 history asli, dan hash figur. Sepuluh checkpoint best cocok dengan sidecar SHA di KC. Hash source sendiri tidak membuktikan eksekusi. Executable B3/B4 tetap eksternal dan belum tracked Git. CAD-ViT reproduction dan seluruh hasil historis tidak diubah.
